In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, WARNING, ERROR, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import

In [ ]:
import copy
import math
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
import matplotlib.gridspec as gridspec

from src.simulation.seasonal_sro import SeasonalSROConfig, simulate_SeasonalSRO
from src.simulation.langevin2d import InfoThermodynLangevin2D
from src.simulation.langevin2d_helper import (
    get_langevin_without_forcing_for_seasonal_SRO,
    InformationThermodynamicsResult,
    calc_diff_bound,
)
from src.analysis.peak_detector import detect_peak_counts_scipy, _fractional_day_of_year
from src.utils.io_pickles import read_pickle, write_pickle


plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")

# Define constants

In [ ]:
ROOT_DIR = str((pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]) / "..").resolve())
DATA_KIND = "KA21"  # "KA21", "V25", "H26"
assert DATA_KIND in ["KA21", "V25", "H26"]

In [ ]:
DEFAULT = SeasonalSROConfig(value_set_name=DATA_KIND)

In [ ]:
df = pd.DataFrame.from_dict(DEFAULT.__dict__, orient="index", columns=["value"])
df.loc["WBJ (Re. Part)", "value"] = DEFAULT.get_bwj_real_part()
df.loc["WBJ (Im. Part)", "value"] = DEFAULT.get_bwj_imaginary_part()
df.loc["Intrinsic Period"] = 2 * math.pi / DEFAULT.get_bwj_imaginary_part()
display(df)
print(np.round(df, 4).to_markdown())

In [ ]:
FIG_DIR = "./fig"
os.makedirs(FIG_DIR, exist_ok=True)

# Simulate Seasonal RO

## Growth Rate

In [ ]:
t_min = 0
t_max = 1  # unit: year
n_timesteps = int(t_max) * 360

dt = (t_max - t_min) / n_timesteps
ts = torch.linspace(t_min, t_max, n_timesteps + 1, dtype=torch.float64)

config = copy.deepcopy(DEFAULT)
# config.phi = 0.0
Rt = config.get_Rt(ts)

plt.rcParams["font.size"] = 18
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(1, 1, 1)

ax.plot(ts * 360, Rt, color="#ABABAB", lw=3)
ax.set_xlabel("Calendar Month")
ax.set_ylabel(r"Growth Rate, $R(t)$ [1/yr]")

ax.set_xticks(np.linspace(0, 360, 13)[:12] + 15, labels=np.arange(1, 13))
ax.set_xlim(0, 360)
for i, t in enumerate(np.linspace(0, 360, 13)):
    if t + 15 > 360:
        break
    ax.axvline(t + 15, color="gray", linestyle="--", alpha=0.3)
ax.axhline(0, color="k", linestyle="-", alpha=1.0)

if (
    DATA_KIND == "KA21WeakNoise"
    or DATA_KIND == "KA21"
    or DATA_KIND == "KA21StrongNoise"
):
    ax.set_ylim(-3, 1)
if DATA_KIND == "V25":
    ax.set_ylim(-4, 2)
if DATA_KIND == "H26":
    ax.set_ylim(-2, 3)

fig.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig01.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig01.pdf", bbox_inches="tight")

del config, t_min, t_max, n_timesteps, dt

## Std and Ensemble Members

In [ ]:
n_batches = 10_000
t_min = 0.0
t_max = 100  # unit: year
n_timesteps = int((t_max - t_min) * 360)
dict_results = {}

for is_seasonality_on in [True, False]:

    config = copy.deepcopy(DEFAULT)
    if not is_seasonality_on:
        config.Ra = 0.0
        seed = 314
    else:
        seed = 42

    # Spin Up
    x0 = torch.tensor([0.0, 10.0], dtype=torch.float64).unsqueeze(0)  # shape (1, 2)
    x0 = x0.repeat(n_batches, 1)  # shape (n_batches, 2)
    # x0 = torch.randn_like(x0) # use this if you want random initial conditions
    _, xs = simulate_SeasonalSRO(
        x0=x0,
        t_min=t_min,
        t_max=t_max,
        n_timesteps=n_timesteps,
        n_batches=n_batches,
        config=config,
        seed=seed,
    )
    x0 = xs[:, -1].clone().detach()
    del xs

    ts, xs = simulate_SeasonalSRO(
        x0=x0,
        t_min=t_min,
        t_max=t_max,
        n_timesteps=n_timesteps,
        n_batches=n_batches,
        config=config,
        seed=seed + 1000,
    )

    dict_results[is_seasonality_on] = {
        "ts": ts.clone().detach(),
        "T": xs[:, :, 0].clone().detach(),
        "h": xs[:, :, 1].clone().detach(),
        "std_T": xs[:, :, 0].std(dim=0),
        "std_h": xs[:, :, 1].std(dim=0),
        "cov": (xs[:, :, 0] * xs[:, :, 1]).mean(dim=0),
    }
    del xs, ts, config

In [ ]:
n = 360 * 5
i_members = {}
if DATA_KIND == "H26":
    i_members["seasonal_on"] = 24  # 3, 5, 24, 33
    i_members["seasonal_off"] = 40  # 25, 33, 39, 40
elif DATA_KIND == "KA21":
    i_members["seasonal_on"] = 24  # 3, 23, 24, 26
    i_members["seasonal_off"] = 39  # 4, 6, 25, 39, 55
else:
    i = 15
    i_members["seasonal_on"] = i
    i_members["seasonal_off"] = i

plt.rcParams["font.size"] = 19
fig, axes = plt.subplots(5, 2, figsize=(14, 15))

for j, is_seasonality_on in enumerate([False, True]):
    T = dict_results[is_seasonality_on]["T"]
    h = dict_results[is_seasonality_on]["h"]
    std_T = dict_results[is_seasonality_on]["std_T"]
    std_h = dict_results[is_seasonality_on]["std_h"]
    cov = dict_results[is_seasonality_on]["cov"]
    ts = dict_results[is_seasonality_on]["ts"]

    if is_seasonality_on:
        i_member = i_members["seasonal_on"]
    else:
        i_member = i_members["seasonal_off"]

    if j == 0:
        sub_ttl = "(a) Steady SRO, $R_a = 0$"
    else:
        sub_ttl = "(b) Unsteady SRO, $R_a = 2.0 |R_0|$"
        if DATA_KIND == "H26":
            sub_ttl = "(b) Unsteady SRO, $R_a = 5.3 |R_0|$"

    for i, (ax, label, color, data) in enumerate(
        zip(
            axes[:, j],
            [
                "$T$ [K]",
                r"std. $T$ [K]",
                "$h$ [m]",
                r"std. $h$ [m]",
                r"cov [K m]",
            ],
            ["#006BA4", "#006BA4", "#FF800E", "#FF800E", "black"],
            [T, std_T, h, std_h, cov],
        )
    ):
        xs = ts.numpy()[-n:] - ts.numpy()[-n]
        ls = "-" if "std" in label else "--"

        if "std" in label or "cov" in label:
            ys = data.numpy()[-n:]
            ax.plot(xs, ys, color=color, linestyle=ls, lw=4)
        else:
            ys = data.numpy()[i_member, -n:]
            ax.plot(xs, ys, color=color, linestyle=ls, lw=2)

        if j == 0:
            ax.set_ylabel(label, fontsize=19)
        if j == 1:
            ax.set_yticklabels([])

        if i == 4:
            ax.set_xlabel("Time [yr]", fontsize=18)

        if "$T$" in label and "std" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(0.5, 1.1, 5))
                ax.set_ylim(0.5, 1.1)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(1.6, 4.0, 5))
                ax.set_ylim(1.6, 4.0)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(1.0, 2.6, 5))
                ax.set_ylim(1.0, 2.6)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(0.4, 1.1, 5))
                ax.set_ylim(0.4, 1.1)
            ax.set_title(r"Standard Deviation (std.): $T$")
        elif "$h$" in label and "std" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(4.9, 5.7, 5))
                ax.set_ylim(4.9, 5.7)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(17, 20, 5))
                ax.set_ylim(17, 20)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(14, 18, 5))
                ax.set_ylim(14, 18)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(5.3, 6.9, 5))
                ax.set_ylim(5.3, 6.9)
            ax.set_title(r"Standard Deviation (std.): $h$")
        elif "$T$" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(-3.0, 3.0, 5))
                ax.set_ylim(-3, 3)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(-10.0, 10.0, 5))
                ax.set_ylim(-10, 10)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(-5, 5.0, 5))
                ax.set_ylim(-5, 5)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(-3, 3.0, 5))
                ax.set_ylim(-3, 3)
            ax.axhline(0, color="k", linestyle="-", alpha=1.0)
            ax.set_title(sub_ttl + "\n\n" + r"One Realization: $T$")
        elif "$h$" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(-20, 20, 5))
                ax.set_ylim(-20, 20)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(-60, 60, 5))
                ax.set_ylim(-60, 60)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(-50, 50, 5))
                ax.set_ylim(-50, 50)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(-20, 20, 5))
                ax.set_ylim(-20, 20)
            ax.axhline(0, color="k", linestyle="-", alpha=1.0)
            ax.set_title(r"One Realization: $h$")
        elif "cov" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks([-0.5, 0.0, 1.0, 2.0])
                ax.set_ylim(-0.5, 2.0)
                pass
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks([-5, 0, 10, 20])
                ax.set_ylim(-5, 20)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(-10, 10, 3))
                ax.set_ylim(-10, 10)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(-5, 0, 3))
                ax.set_ylim(-5, 0)
            ax.axhline(0, color="k", linestyle="-", alpha=1.0)
            ax.set_title(r"Covariance: $\left\langle T h \right\rangle$")

        for yr in np.linspace(0, 5, 11):
            ls = "-" if yr in [0, 1, 2, 3, 4, 5] else "--"
            ax.axvline(yr, color="gray", linestyle=ls, alpha=0.3)
        ax.set_xlim(0, 5)

plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig02.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig02.pdf", bbox_inches="tight")
elif DATA_KIND == "H26":
    fig.savefig(f"{FIG_DIR}/fig07.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig07.pdf", bbox_inches="tight")
del T, h, std_T, std_h, ts, xs

# TUR

In [ ]:
initial_month = 3  # or 9, results are almost independent of initial_month
assert 1 <= initial_month <= 12

t_min = float(initial_month - 1) / 12.0
t_max = 200  # unit: year
nt_year = 360  # number of days in a year
n_timesteps = int((t_max - t_min) * nt_year)

dict_results = {}
lst_amp_Ra = [0.5, 1.0, 2.0]
if DATA_KIND == "H26":
    lst_amp_Ra = [1.0, 3.0, 6.0]

for amp_Ra in lst_amp_Ra:
    logger.info(f"Calculating for {amp_Ra=}")

    config = copy.deepcopy(DEFAULT)
    config.Ra = abs(config.R0) * amp_Ra

    mu0 = torch.tensor([0.0, 10.0], dtype=torch.float64)
    # results are almost independent of mu0
    # mu0 = torch.randn_like(mu0)  # use this for random initialization

    ts, langevin = get_langevin_without_forcing_for_seasonal_SRO(
        cfg=config, t_min=t_min, t_max=t_max, n_timesteps=n_timesteps, mu0=mu0
    )
    result = InformationThermodynamicsResult(langevin)

    mean_abs_T = torch.mean(langevin.x_ave()[-nt_year - 1 : -1].abs()).item()
    assert mean_abs_T < 1e-7  # mean is almost zero

    dict_results[amp_Ra] = {
        "month": np.linspace(0, 12, nt_year, endpoint=False),
        "var_T": (langevin.x2_ave() - langevin.x_ave() ** 2)[-nt_year - 1 : -1],
        "Rt": config.get_Rt(ts)[-nt_year - 1 : -1],
        "tur_bound_for_dvar_dt_sq": torch.sqrt(result.bound_dvar_dt_sq[-nt_year:]),
        "dvar_dt_sq": torch.sqrt(result.dvar_dt_sq[-nt_year:]),  # dx_dt_cdot_x
        "sum_of_all": (result.sX + result.qX - result.iflX)[-nt_year:].clone().detach(),
        "Q": result.qX[-nt_year:].clone().detach(),
        "S": result.sX[-nt_year:].clone().detach(),
        "-I": -result.iflX[-nt_year:].clone().detach(),
        "P": result.tur_pt[-nt_year:].clone().detach(),
        "current": result.tur_current[-nt_year:].clone().detach(),
        "langevin": langevin,
        "all_results": result,
        "ts": ts,
    }
    del config, mu0, ts, langevin, result, mean_abs_T

In [ ]:
plt.rcParams["font.size"] = 19
fig, all_axes = plt.subplots(4, 3, sharex=False, figsize=(17, 15))

lst_amp_Ra = [0.5, 1.0, 2.0]
if DATA_KIND == "H26":
    lst_amp_Ra = [1.0, 3.0, 6.0]

for j, amp_Ra in enumerate(lst_amp_Ra):
    axes = all_axes[:, j]
    d = dict_results[amp_Ra]

    ax = axes[0]
    ax.plot(d["month"], d["Rt"], color="#ABABAB", lw=2)
    if j == 0:
        ax.set_ylabel(r"$R(t)$ [1/yr]")
    ax.axhline(0.0, ls="-", color="k", lw=1.0)
    if (
        DATA_KIND == "KA21WeakNoise"
        or DATA_KIND == "KA21"
        or DATA_KIND == "KA21StrongNoise"
    ):
        ax.set_ylim(-3, 1)
    if DATA_KIND == "V25":
        ax.set_ylim(-4, 2)
    if DATA_KIND == "H26":
        ax.set_ylim(-2, 3)

    if j == 0:
        if DATA_KIND == "H26":
            assert amp_Ra == 1.0
            ax.set_title(r"(a) $R_a = 1.0 |R_0|$" + "\n\n" + "Growth Rate, $R(t)$")
        else:
            assert amp_Ra == 0.5
            ax.set_title(r"(a) $R_a = 0.5 |R_0|$" + "\n\n" + "Growth Rate, $R(t)$")
    elif j == 1:
        if DATA_KIND == "H26":
            assert amp_Ra == 3.0
            ax.set_title(r"(b) $R_a = 3.0 |R_0|$" + "\n\n" + "Growth Rate, $R(t)$")
        else:
            assert amp_Ra == 1.0
            ax.set_title(r"(b) $R_a = 1.0 |R_0|$" + "\n\n" + "Growth Rate, $R(t)$")
    elif j == 2:
        if DATA_KIND == "H26":
            assert amp_Ra == 6.0
            ax.set_title(r"(c) $R_a = 6.0 |R_0|$" + "\n\n" + "Growth Rate, $R(t)$")
        else:
            assert amp_Ra == 2.0
            ax.set_title(r"(c) $R_a = 2.0 |R_0|$" + "\n\n" + "Growth Rate, $R(t)$")

    color_T = "#006BA4"
    ax = axes[1]
    ax.plot(d["month"], d["var_T"], color=color_T, lw=3)
    if j == 0:
        ax.set_ylabel(r"var. $T$ [K$^2$]")
    if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
        ax.set_ylim(0.1, 1.1)
        ax.set_yticks(np.linspace(0.1, 1.1, 6))
    if DATA_KIND == "KA21StrongNoise":
        ax.set_ylim(2, 14)
        ax.set_yticks(np.linspace(2, 14, 7))
    if DATA_KIND == "V25":
        ax.set_ylim(0, 7)
        ax.set_yticks(np.linspace(0, 7, 8))
    if DATA_KIND == "H26":
        ax.set_ylim(0.2, 1.2)
        ax.set_yticks(np.linspace(0.2, 1.2, 6))
    ax.set_title(r"Variance of $T$ ($= \langle T^2\rangle$)")

    ax = axes[2]
    l = r"$\dot{\mathcal{P}}$ ($=\frac{2\dot{\mathcal{Q}}}{(\sigma^T)^2}+\frac{{\rm d}\mathcal{S}}{{\rm d}t}-\dot{\mathcal{I}}$)"
    ax.plot(d["month"], d["P"], color="#C85200", ls="--", label=l, lw=2)
    print(f'{torch.min(d["P"])=}')
    l = r"$\left(\frac{1}{2}(\sigma^T)^2\langle T^2\rangle \right)^{-1} \left(\frac{1}{2}\frac{{\rm d}}{{\rm d}t} \langle T^2 \rangle\right)^2$"
    ax.plot(d["month"], d["current"], color="k", label=l, lw=3)
    print(f'{torch.min(d["current"])=}')
    if j == 0:
        ax.set_ylabel(r"Change Rate [1/yr]")
    if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "KA21StrongNoise":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "V25":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "H26":
        ax.set_ylim(-5.0, 30.0)
        ax.set_yticks(np.linspace(-5, 30, 8))
    ax.set_title("TUR")
    ax.axhline(0.0, ls="-", color="k", lw=1.0)
    ratio = d["current"] / d["P"]
    i_max, i_min = int(torch.argmax(ratio)), int(torch.argmin(ratio))
    print(
        f"{DATA_KIND}, {amp_Ra=}: ratio max {float(ratio[i_max].item()):.4f} in month {int(d['month'][i_max]) + 1}, "
        f"min {float(ratio[i_min].item()):.4f} in month {int(d['month'][i_min]) + 1}\n"
    )
    ax_ratio = ax.twinx()
    ratio_color = "#006BA4"
    ax_ratio.plot(
        d["month"],
        ratio,
        color=ratio_color,
        ls=":",
        lw=3,
        label="Ratio (black solid / red dashed)",
    )
    ax_ratio.set_ylim(-0.25, 1.0)  # zero level with the left axis (-5 to 20)
    ax_ratio.set_yticks(np.linspace(0.0, 1.0, 5))
    ax_ratio.tick_params(axis="y", colors=ratio_color)
    if j == len(lst_amp_Ra) - 1:
        ax_ratio.set_ylabel("Ratio", color=ratio_color)
    else:
        ax_ratio.set_yticklabels([])
    if j == 0:
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax_ratio.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, ncol=1, fontsize=14)

    ax = axes[3]
    # l = r"$\dot{\mathcal{P}}_{T}$ ($=\frac{{\rm d}\mathcal{S}_T}{{\rm d}t}+\frac{2\dot{\mathcal{Q}}_T}{\sigma_T^2}-\dot{\mathcal{I}}_T$)"
    # ax.plot(d["month"], d["sum_of_all"], color="r", ls="--", label=l, lw=2)
    ax.plot(
        d["month"],
        d["Q"],
        label=r"$\frac{2\dot{\mathcal{Q}}}{(\sigma^T)^2}$",
        ls=":",
        lw=3,
        color="#FFBC79",
    )
    ax.plot(
        d["month"],
        d["S"],
        label=r"$\frac{{\rm d}\mathcal{S}}{{\rm d}t}$",
        ls="--",
        lw=3,
        color="#A2C8EC",
    )
    ax.plot(d["month"], d["-I"], label=r"$-\dot{\mathcal{I}}$", lw=3, color="#C85200")
    ax.axhline(0.0, ls="-", color="k", lw=0.5)
    if j == 0:
        ax.set_ylabel("Change Rate [1/yr]")
    if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "KA21StrongNoise":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "V25":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "H26":
        ax.set_ylim(-5.0, 30.0)
        ax.set_yticks(np.linspace(-5, 30, 8))
    ax.set_xlabel("Calendar Month")
    ax.set_title("TUR Terms")
    if j == 0:
        ax.legend(ncol=3, fontsize=18)
    if j != 0:
        for ax in axes.flatten():
            ax.set_yticklabels([])

    for ax in axes.flatten():
        ax.set_xlim(0, 12)
        ax.set_xticks(
            np.linspace(1, 12, 12) - 0.5, np.linspace(1, 12, 12, dtype=np.int32)
        )
        # ax.set_xticklabels([])
    # axes.flatten()[-1].set_xticks(
    #     np.linspace(1, 12, 12) - 0.5, np.linspace(1, 12, 12, dtype=np.int32)
    # )

plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig03.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig03.pdf", bbox_inches="tight")

In [ ]:
nt_year = 360  # number of days in a year
plt.rcParams["font.size"] = 19
fig, all_axes = plt.subplots(4, 3, sharey=True, sharex=False, figsize=(17, 15))

lst_amp_Ra = [0.5, 1.0, 2.0]
if DATA_KIND == "H26":
    lst_amp_Ra = [1.0, 3.0, 6.0]

for j, amp_Ra in enumerate(lst_amp_Ra):
    axes = all_axes[:, j]
    d = dict_results[amp_Ra]
    langevin = d["langevin"]

    ax = axes[0]
    if j == 0:
        if DATA_KIND == "H26":
            assert amp_Ra == 1.0
            ax.set_title(
                r"(a) $R_a = 1.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
        else:
            assert amp_Ra == 0.5
            ax.set_title(
                r"(a) $R_a = 0.5 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
    elif j == 1:
        if DATA_KIND == "H26":
            assert amp_Ra == 3.0
            ax.set_title(
                r"(b) $R_a = 3.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
        else:
            assert amp_Ra == 1.0
            ax.set_title(
                r"(b) $R_a = 1.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
    elif j == 2:
        if DATA_KIND == "H26":
            assert amp_Ra == 6.0
            ax.set_title(
                r"(c) $R_a = 6.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
        else:
            assert amp_Ra == 2.0
            ax.set_title(
                r"(c) $R_a = 2.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )

    q = (langevin.Qx_over_Tx())[-nt_year:]

    a = (langevin.a11**2) * langevin.x2_ave() / langevin.Tx
    a = a[-nt_year - 1 : -1]

    b = 2 * langevin.xy_ave() * (langevin.a11 * langevin.a12) / langevin.Tx
    b = b[-nt_year - 1 : -1]

    c = (langevin.a12**2) * langevin.y2_ave() / langevin.Tx
    c = c[-nt_year - 1 : -1]

    e = langevin.a11
    e = e[-nt_year - 1 : -1]

    s = a + b + c + e

    for i, (l, data) in enumerate(zip(["A", "B", "C", "D"], [a, b, c, e])):

        ax = axes[i]
        label = r"$\frac{2\dot{\mathcal{Q}}}{(\sigma^T)^2}=A+B+C+D$"
        ax.plot(d["month"], q, label=label, ls=":", lw=3, color="#FFBC79")
        ax.plot(d["month"], data, color="k", lw=2, label=l)
        # ax.plot(d["month"], s, color="gray")

        if j == 0:
            ax.legend(ncol=2, fontsize=18, loc="upper right")
        if i == 1:
            ax.set_title(
                r"$B = (2/(\sigma^T)^2) 2 R(t) F_1 \left\langle Th \right\rangle$"
            )
        elif i == 2:
            ax.set_title(
                r"$C = (2/(\sigma^T)^2) (F_1)^2 \left\langle h^2 \right\rangle$"
            )
        elif i == 3:
            ax.set_title(r"$D = R(t)$")

        if j == 0:
            ax.set_ylabel("Change Rate [1/yr]")

        if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
            ax.set_ylim(-5.0, 20.0)
            ax.set_yticks(np.linspace(-5, 20, 6))
        if DATA_KIND == "KA21StrongNoise":
            ax.set_ylim(-5.0, 20.0)
            ax.set_yticks(np.linspace(-5, 20, 6))
        if DATA_KIND == "V25":
            ax.set_ylim(-5.0, 20.0)
            ax.set_yticks(np.linspace(-5, 20, 6))
        if DATA_KIND == "H26":
            ax.set_ylim(-10.0, 40.0)
            ax.set_yticks([-10, 0, 10, 20, 30, 40])

    ax = axes[-1]
    ax.set_xlabel("Calendar Month")
    for ax in axes.flatten():
        ax.axhline(0.0, ls="-", color="k", lw=0.5)
        ax.set_xlim(0, 12)
        ax.set_xticks(
            np.linspace(1, 12, 12) - 0.5, np.linspace(1, 12, 12, dtype=np.int32)
        )
plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig05.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig05.pdf", bbox_inches="tight")
elif DATA_KIND == "H26":
    fig.savefig(f"{FIG_DIR}/fig08.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig08.pdf", bbox_inches="tight")

# Dependence on Ra

In [ ]:
peak_mode = "all_peaks"  # "max_only", "all_peaks"
t_max = 10_000  # year

is_random_init = False
n_months_for_running_mean = 5  # 3 or 5
threshold_n_months = 6  # 7 or 6

dk = DATA_KIND
out_dir = f"{ROOT_DIR}/data/models/SRO_long_run_{t_max:06}yr_{dk}"
if is_random_init:
    out_dir += "_random_init"

lst_amp_Ra = np.linspace(0, 2.5, 26) if dk != "H26" else np.linspace(0, 6.0, 61)
n_per_month = 30  # days in a month
nt_year = 12 * n_per_month  # days in a year
peak_counts = np.zeros(shape=(len(lst_amp_Ra), nt_year), dtype=np.int32)
lst_bounds = []

for idx, amp in tqdm(enumerate(lst_amp_Ra), total=len(lst_amp_Ra)):
    out_file_path = f"{out_dir}/SRO_Ra{str(amp).replace('.', 'p')[:3]}.pickle"
    data = read_pickle(out_file_path)

    years = data["ts"]
    time_series = pd.Series(data["xs"][0, :, 0])  # SST time series
    assert len(years) == len(time_series)

    smoothed = time_series.rolling(
        window=n_months_for_running_mean * n_per_month + 1, center=True, win_type=None
    ).mean()

    logger.setLevel(WARNING)

    elnino_counts = detect_peak_counts_scipy(
        years=years,
        smoothed_series=smoothed,
        std_value=float(np.std(time_series)),
        n_per_month=n_per_month,
        mode="elnino",
        peak_mode=peak_mode,
        peak_min_distance=int(n_per_month * n_months_for_running_mean),
        peak_min_width=int(n_per_month * n_months_for_running_mean),
        threshold_n_months=threshold_n_months,
    )
    assert len(elnino_counts) == nt_year

    peak_counts[idx] = elnino_counts

    config = copy.deepcopy(DEFAULT)
    config.Ra = abs(config.R0) * amp
    diff_bound, _, min_bound = calc_diff_bound(
        t_min=0.0, t_max=200.0, nt_year=nt_year, config=config
    )
    lst_bounds.append(diff_bound)

    logger.setLevel(INFO)
    del data, years, time_series, smoothed
    del elnino_counts, diff_bound, min_bound, config

In [ ]:
width = 10  # number of days in each bin

results, centers = [], []
days = np.arange(0, 360 + width, width)
assert days[0] == 0 and days[-1] == 360

for start in days:
    end = start + width
    if end > 360:
        break
    data = peak_counts[:, start:end]
    assert data.shape == (len(lst_amp_Ra), width), f"{data.shape=}, {start=}, {end=}"
    results.append(np.sum(data, axis=1))
    centers.append((start + end) // 2 + 1)

peaks = np.stack(results, axis=1)
probs = peaks / np.sum(peaks, axis=1, keepdims=True)
diff_probs = np.max(probs, axis=1) - np.min(probs, axis=1)
assert np.all(np.abs(np.sum(probs, axis=1) - 1.0) < 1e-7)

xs = centers
ys = lst_amp_Ra
X, Y = np.meshgrid(xs, ys, indexing="ij")

plt.rcParams["font.size"] = 14
fig = plt.figure(figsize=(13, 6))
gs = gridspec.GridSpec(1, 5, figure=fig)

ax = fig.add_subplot(gs[0, 1:3])

ret = ax.pcolormesh(
    X, Y, probs.transpose(), shading="nearest", cmap="magma", vmin=0.0, vmax=0.12
)
cbar = fig.colorbar(ret, ax=ax, extend="max")
ax.set_title("Relative Frequency")

# ax.set_ylabel(f"Amplitude of $R_a$ ($= |R_a/R_0|$)")
ax.set_ylim(min(lst_amp_Ra) - 0.05, max(lst_amp_Ra) + 0.05)
ax.set_xticks(
    np.linspace(0, 360, 13)[:12] + 15,
    np.arange(1, 13),
)
ax.set_xlabel("Calendar Month")
# ax.set_yticks([])

if DATA_KIND == "H26":
    ax.axhline(5.33, color="w", ls="--")
else:
    ax.axhline(2.0, color="w", ls="--")

ax = fig.add_subplot(gs[0, 0:1])
ax.plot(lst_bounds, lst_amp_Ra, color="k")
print(min(lst_bounds))
ax.set_ylim(min(lst_amp_Ra) - 0.05, max(lst_amp_Ra) + 0.05)
ax.set_xlabel(
    r"$\Delta \dot{\cal P}$ ($= \max \;\dot{\cal P} - \min \;\dot{\cal P}$) [1/yr]"
)
ax.set_ylabel(f"Magnitude of $R_a$ ($= |R_a/R_0|$)")
if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
    ax.set_xticks([0, 5, 10, 15])
    ax.set_xlim(-0.5, 16.0)
elif DATA_KIND == "H26":
    ax.set_xticks([0, 5, 10, 15, 20, 25])
    ax.set_xlim(-0.5, 25.0)
elif DATA_KIND == "V25":
    ax.set_xticks([0, 5, 10, 15, 20])
    ax.set_xlim(-0.5, 21.0)

ax = fig.add_subplot(gs[0, 3:4])
ax.plot(np.sum(peaks, axis=1) / t_max, lst_amp_Ra, color="k")
ax.set_ylim(min(lst_amp_Ra) - 0.05, max(lst_amp_Ra) + 0.05)
ax.set_xlabel(r"Num. of Peaks [1/yr]")
# ax.set_yticks([])
if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
    ax.set_xticks([0.12, 0.125, 0.130])
    ax.set_xlim(0.12, 0.13)
if DATA_KIND == "KA21StrongNoise":
    ax.set_xticks([0.12, 0.125, 0.130])
    ax.set_xlim(0.12, 0.13)
if DATA_KIND == "V25":
    ax.set_xticks([0.11, 0.12, 0.130])
    ax.set_xlim(0.11, 0.13)
if DATA_KIND == "H26":
    ax.set_xticks([0.12, 0.125, 0.130])
    ax.set_xlim(0.12, 0.13)
plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig04.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig04.pdf", bbox_inches="tight")

# Fixed Ra with R0, epsilon, or F2 varied

In [ ]:
COV_DIR = f"{ROOT_DIR}/data/processed/revision1_covariance"
PEAK_DIR = f"{ROOT_DIR}/data/processed/revision1_peaks"
RESULT_DIR = f"{ROOT_DIR}/data/processed/revision1_results"
for d in (COV_DIR, PEAK_DIR, RESULT_DIR):
    os.makedirs(d, exist_ok=True)

NT_YEAR = 360
T_MAX_COV = 200.0
T_MAX_SDE = 50_000
T_MIN_FIG4 = 0.0

In [ ]:
def load_tur(
    config: SeasonalSROConfig, t_min: float, path: str
) -> tuple[torch.Tensor, InfoThermodynLangevin2D, InformationThermodynamicsResult]:
    #
    n_timesteps = int((T_MAX_COV - t_min) * NT_YEAR)
    if os.path.exists(path):
        saved = read_pickle(path)
        assert saved["config"] == config and saved["t_min"] == t_min
        ts = saved["ts"]
        logger.setLevel(WARNING)
        langevin = InfoThermodynLangevin2D(
            dt=(T_MAX_COV - t_min) / n_timesteps,
            mu=saved["mu"],
            sigma=saved["sigma"],
            As=config.mat_A(ts),
            Bs=config.mat_B(ts),
            fs=torch.zeros(n_timesteps + 1, 2, dtype=torch.float64),
        )
        logger.setLevel(INFO)
    else:
        mu0 = torch.tensor([0.0, 10.0], dtype=torch.float64)
        ts, langevin = get_langevin_without_forcing_for_seasonal_SRO(
            cfg=config, t_min=t_min, t_max=T_MAX_COV, n_timesteps=n_timesteps, mu0=mu0
        )
        write_pickle(
            {
                "config": copy.deepcopy(config),
                "t_min": t_min,
                "ts": ts,
                "mu": langevin.mu,
                "sigma": langevin.sigma,
            },
            path,
        )
    mean_abs_T = torch.mean(langevin.x_ave()[-NT_YEAR - 1 : -1].abs()).item()
    assert mean_abs_T < 1e-7  # the mean has decayed, as asserted in the TUR cells
    return ts, langevin, InformationThermodynamicsResult(langevin)


def final_year(
    langevin: InfoThermodynLangevin2D, result: InformationThermodynamicsResult
) -> dict[str, np.ndarray]:
    #
    var_T = (langevin.x2_ave() - langevin.x_ave() ** 2)[-NT_YEAR - 1 : -1].numpy()
    P = result.tur_pt[-NT_YEAR:].numpy()
    rhs = result.tur_current[-NT_YEAR:].numpy()
    return {"var_T": var_T, "P": P, "rhs": rhs, "ratio": rhs / P}

In [ ]:
# Factors 2**(k/8), k = -16..16 (0.25 to 4); R0 shifts -2.4 to +0.8 /yr in steps of 0.1 /yr.
# factor 1 is the base case
FACTORS = [2.0 ** (k / 8) for k in range(-16, 17) if k != 0]
# 1/yr; shift 0 is the base case
R0_SHIFTS = [round(0.1 * i, 1) for i in range(-24, 9) if i != 0]
# Wider epsilon sweep, 2**(k/4) = 0.0625-0.21 and 4.76-9.51
EPS_FACTORS_EXT = [2.0 ** (k / 4) for k in [*range(-16, -8), *range(9, 14)]]


def fixed_ra_cases() -> list[dict]:
    cases = []
    for kind in ["KA21"]:
        base = SeasonalSROConfig(value_set_name=kind)
        variants = [("base", 0.0, "base", copy.deepcopy(base))]
        for d in R0_SHIFTS:
            cfg = copy.deepcopy(base)
            cfg.R0 = base.R0 + d
            label = (
                f"R0{d:+.1f}".replace("+", "plus")
                .replace("-", "minus")
                .replace(".", "p")
            )
            variants.append(("R0", d, label, cfg))
        for attr in ["epsilon", "F2"]:
            factors = FACTORS
            if attr == "epsilon":
                factors = FACTORS + EPS_FACTORS_EXT
            for f in factors:
                cfg = copy.deepcopy(base)
                setattr(cfg, attr, getattr(base, attr) * f)
                variants.append((attr, f, f"{attr}x{f:.2f}".replace(".", "p"), cfg))
        for family, change, label, cfg in variants:
            cases.append(
                {
                    "data_kind": kind,
                    "family": family,
                    "change": change,
                    "label": label,
                    "config": cfg,
                    "Ra_fixed": abs(base.Ra),
                    # simulate_SeasonalSRO_and_write_out_results sets Ra = |R0| * amp_Ra
                    "amp_Ra": abs(base.Ra) / abs(cfg.R0),
                    "path": f"{ROOT_DIR}/data/models/fixed_Ra_SRO_long_run_{T_MAX_SDE:06}yr_{kind}/SRO_{label}.pickle",
                }
            )
    assert len(cases) == 110 and len({c["path"] for c in cases}) == len(cases)
    for c in cases:
        cfg = c["config"]
        assert cfg.R0 < cfg.epsilon  # damped
        assert cfg.F1 * cfg.F2 > (cfg.R0 + cfg.epsilon) ** 2 / 4  # oscillatory
    return cases

In [ ]:
def get_diagnostics(config: SeasonalSROConfig, cov_path: str) -> dict:
    #
    _, langevin, result = load_tur(config, T_MIN_FIG4, cov_path)
    q = final_year(langevin, result)
    return {
        "R0": config.R0,
        "Ra": config.Ra,
        "bwj_im": config.get_bwj_imaginary_part(),
        "dP": q["P"].max() - q["P"].min(),
    }


def detect_peaks(years: np.ndarray, time_series: pd.Series, peak_mode: str) -> dict:
    #
    n_per_month, n_months_for_running_mean = 30, 5
    smoothed = time_series.rolling(
        window=n_months_for_running_mean * n_per_month + 1, center=True, win_type=None
    ).mean()
    logger.setLevel(WARNING)
    counts, peak_indices, _, masked_years = detect_peak_counts_scipy(
        years=years,
        smoothed_series=smoothed,
        std_value=float(np.std(time_series)),
        n_per_month=n_per_month,
        mode="elnino",
        peak_mode=peak_mode,
        debug=True,
        peak_min_distance=int(n_per_month * n_months_for_running_mean),
        peak_min_width=int(n_per_month * n_months_for_running_mean),
        threshold_n_months=6,
    )
    logger.setLevel(INFO)
    peak_years = np.asarray(masked_years)[np.asarray(peak_indices, dtype=int)]
    days = _fractional_day_of_year(peak_years, days_per_year=NT_YEAR)
    assert np.array_equal(np.bincount(days, minlength=NT_YEAR + 1)[1:], counts)
    return {"peak_years": peak_years, "days": days}


def concentration(days: np.ndarray) -> float:
    #
    counts = np.bincount(days - 1, minlength=NT_YEAR)
    theta = 2.0 * np.pi * np.arange(NT_YEAR) / NT_YEAR
    return float(np.abs(np.sum(counts * np.exp(1j * theta))) / counts.sum())


def analyze_run(run_path: str, stem: str) -> dict:
    #
    data = read_pickle(run_path)
    config = data["config"]
    years = data["ts"]
    time_series = pd.Series(data["xs"][0, :, 0])
    del data
    row = get_diagnostics(config, f"{COV_DIR}/fig04setting_{stem}.pickle")
    path = f"{PEAK_DIR}/{stem}_all_peaks.pickle"
    if os.path.exists(path):
        peaks = read_pickle(path)
    else:
        peaks = detect_peaks(years, time_series, "all_peaks")
        write_pickle(peaks, path)
    row["rbar"] = concentration(peaks["days"])
    return row

In [ ]:
rows = []
for c in tqdm(fixed_ra_cases()):
    row = {"data_kind": c["data_kind"], "family": c["family"], "change": c["change"]}
    row.update(analyze_run(c["path"], f"{c['data_kind']}_{c['label']}"))
    assert abs(row["Ra"] - c["Ra_fixed"]) < 1e-9 and row["R0"] == c["config"].R0
    rows.append(row)
df_fixed = pd.DataFrame(rows)
df_fixed.to_csv(f"{RESULT_DIR}/fixed_Ra_test.csv", index=False)

sweep_dir = f"{ROOT_DIR}/data/models/SRO_long_run_{T_MAX_SDE:06}yr_KA21"
rows = []
for amp in tqdm(np.linspace(0, 2.5, 26)):
    tag = str(amp).replace(".", "p")[:3]
    row = {"data_kind": "KA21", "family": "Ra_sweep", "change": amp}
    row.update(analyze_run(f"{sweep_dir}/SRO_Ra{tag}.pickle", f"KA21_Ra_sweep_{tag}"))
    rows.append(row)
df_sweep = pd.DataFrame(rows)
df_sweep.to_csv(f"{RESULT_DIR}/Ra_sweep_KA21.csv", index=False)

dP_fig4, _, _ = calc_diff_bound(
    t_min=0.0,
    t_max=200.0,
    nt_year=NT_YEAR,
    config=SeasonalSROConfig(value_set_name="KA21"),
)
assert (
    abs(dP_fig4 - df_sweep.loc[np.isclose(df_sweep["change"], 2.0), "dP"].item())
    < 1e-10
)

display(df_fixed.round(4))
display(df_sweep.round(4))

In [ ]:
def plot_fig_e1(metric: str, ylabel: str, out_stem: str, kind: str = r"'KA21'") -> None:
    #
    period = lambda df: 2.0 * np.pi / df["bwj_im"]
    df_f = (
        pd.read_csv(f"{RESULT_DIR}/fixed_Ra_test.csv")
        .query(f"data_kind == {kind}")
        .assign(bwj_period=period)
    )
    df_s = pd.read_csv(f"{RESULT_DIR}/Ra_sweep_KA21.csv").assign(bwj_period=period)
    styles = {
        "R0": ("v", "#006BA4", r"$R_0$ varied", 7.0),
        "epsilon": ("^", "#CFCFCF", r"$\varepsilon$ varied", 7.0),
        "F2": ("D", "#FF800E", r"$F_2$ varied", 6.0),
    }
    panels = [
        (
            "dP",
            r"$\Delta \dot{\cal P}$ ($= \max \;\dot{\cal P} - \min \;\dot{\cal P}$) [1/yr]",
            r"(a) $\Delta \dot{\cal P}$",
        ),
        (
            "bwj_period",
            r"$2\pi$ / Im Part of BWJ Index [yr]",
            r"(b) Oscillation Period",
        ),
    ]
    plt.rcParams["font.size"] = 14
    fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
    for ax, (key, xlabel, title) in zip(axes, panels):
        ax.plot(df_s[key], df_s[metric], color="k", lw=1)
        ax.errorbar(
            df_s[key],
            df_s[metric],
            # yerr=df_s[f"{metric}_se"],
            fmt="o",
            color="k",
            ms=4,
            lw=3,
            label=r"$R_a$ varied",
        )
        for family, (marker, color, label, ms) in styles.items():
            sel = df_f[df_f["family"] == family]
            ax.errorbar(
                sel[key],
                sel[metric],
                # yerr=sel[f"{metric}_se"],
                fmt=marker,
                color=color,
                ms=ms,
                mec="k",
                mew=0.1,
                label=label,
                lw=0.5,
            )
        base = df_f[df_f["family"] == "base"]
        ax.errorbar(
            base[key],
            base[metric],
            fmt="o",
            mfc="none",
            mec="k",
            ms=11,
            label="Base",
            lw=5,
            mew=3.0,
        )
        ax.set_xlabel(xlabel)
        ax.set_title(title)
    axes[0].set_ylabel(ylabel)
    axes[0].legend(fontsize=12.5)
    plt.tight_layout()
    plt.show()
    fig.savefig(f"{FIG_DIR}/{out_stem}.pdf", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/{out_stem}.jpg", bbox_inches="tight")


plot_fig_e1("rbar", "Peak Concentration", "fig09", kind=r"'KA21'")

In [ ]:
f2_cases = [
    c
    for c in fixed_ra_cases()
    if c["data_kind"] == "KA21" and c["family"] in ("base", "F2")
]
f2_factors = np.array([1.0 if c["family"] == "base" else c["change"] for c in f2_cases])
order = np.argsort(f2_factors)
f2_cases, f2_factors = [f2_cases[i] for i in order], f2_factors[order]
assert len(f2_cases) == 33

df_ka21 = pd.read_csv(f"{RESULT_DIR}/fixed_Ra_test.csv").query("data_kind == 'KA21'")
f2_dP, f2_dP_var = [], []
for c in f2_cases:
    sel = df_ka21[
        (df_ka21["family"] == c["family"]) & np.isclose(df_ka21["change"], c["change"])
    ]
    assert len(sel) == 1
    f2_dP.append(sel["dP"].item())
    cov_path = f"{COV_DIR}/fig04setting_KA21_{c['label']}.pickle"
    _, langevin, result = load_tur(
        read_pickle(cov_path)["config"], T_MIN_FIG4, cov_path
    )
    q = final_year(langevin, result)
    assert abs((q["P"].max() - q["P"].min()) - f2_dP[-1]) < 1e-10
    f2_dP_var.append(q["rhs"].max() - q["rhs"].min())
f2_share = np.array(f2_dP_var) / np.array(f2_dP)  # share of Delta P carried by P_var

width = 10
f2_counts = np.zeros((len(f2_cases), NT_YEAR), dtype=np.int64)
for idx, c in enumerate(f2_cases):
    days = read_pickle(f"{PEAK_DIR}/KA21_{c['label']}_all_peaks.pickle")["days"]
    f2_counts[idx] = np.bincount(days - 1, minlength=NT_YEAR)
f2_peaks = f2_counts.reshape(len(f2_cases), NT_YEAR // width, width).sum(axis=2)
f2_probs = f2_peaks / np.sum(f2_peaks, axis=1, keepdims=True)
assert np.all(np.abs(np.sum(f2_probs, axis=1) - 1.0) < 1e-7)
centers = np.arange(0, NT_YEAR, width) + width // 2 + 1

ys = np.log2(f2_factors)
y_lim = (ys.min() - 0.1, ys.max() + 0.1)
y_ticks, y_labels = np.log2([0.25, 0.5, 1.0, 2.0, 4.0]), ["0.25", "0.5", "1", "2", "4"]

plt.rcParams["font.size"] = 14
fig = plt.figure(figsize=(13, 6))
gs = fig.add_gridspec(1, 5)

ax = fig.add_subplot(gs[0, 1:3])
X, Y = np.meshgrid(centers, ys, indexing="ij")
ret = ax.pcolormesh(
    X, Y, f2_probs.transpose(), shading="nearest", cmap="magma", vmin=0.0, vmax=0.16
)
print(np.max(f2_probs))
fig.colorbar(ret, ax=ax, extend=None)
ax.set_title("(b) Relative Frequency")
ax.set_xticks(np.linspace(0, 360, 13)[:12] + 15, np.arange(1, 13))
ax.set_xlabel("Calendar Month")
ax.axhline(0.0, color="w", ls="--")
ax.set_ylim(*y_lim)
ax.set_yticks(y_ticks, y_labels)

ax = fig.add_subplot(gs[0, 0:1])
ax.plot(f2_dP, ys, color="k")
ax.set_xlabel(
    r"$\Delta \dot{\cal P}$ ($= \max \;\dot{\cal P} - \min \;\dot{\cal P}$) [1/yr]"
)
ax.set_ylabel(r"Factor of $F_2$ ($= F_2 / F_2^{\rm KA21}$)")
ax.set_title(r"(a) $\Delta \dot{\cal P}$")
ax.set_xticks([0, 10, 20, 30, 40, 50])
ax.set_xlim(-0.5, 50.0)
ax.set_ylim(*y_lim)
ax.set_yticks(y_ticks, y_labels)

ax = fig.add_subplot(gs[0, 3:4])
ax.plot(f2_share, ys, color="k")
ax.set_xlabel(r"$\Delta \dot{\cal P}_{\rm var} / \Delta \dot{\cal P}$")
ax.set_title(r"(c) $\Delta \dot{\cal P}_{\rm var} / \Delta \dot{\cal P}$")
ax.set_xticks([0.5, 0.75, 1.0])
ax.set_xlim(0.5, 1.0)
ax.set_ylim(*y_lim)
ax.set_yticks(y_ticks, y_labels)

plt.tight_layout()
plt.show()
fig.savefig(f"{FIG_DIR}/fig10.jpg", bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/fig10.pdf", bbox_inches="tight")